# DTMIR-Pro+ — quick visual results (dehaze + derain)

This notebook trains the **winning config only (B1)** — trainable translator — on
**SOTS dehaze** and **Rain13K derain**, then produces:
- side-by-side images:  degraded | restored | clean
- metrics:  PSNR, SSIM, **LPIPS**, **FID**

It is intentionally fast (one config, few epochs).

## 1. GPU + dependencies

In Kaggle: **Settings → Accelerator → GPU T4** (then restart the session).

In [ ]:
import shutil, subprocess, torch
if shutil.which('nvidia-smi'):
    print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                 capture_output=True, text=True).stdout.strip())
else:
    print('No GPU found — enable GPU accelerator and re-run.')
print('CUDA:', torch.cuda.is_available())

In [ ]:
!pip install -q torch torchvision torchmetrics torch-fidelity huggingface_hub pyyaml pillow numpy

## 2. Download the code

Paste your fresh `dtmir_pro_plus.zip` link into `CODE_URL`.

In [ ]:
import os, zipfile, subprocess

CODE_URL = "PASTE_NEW_ZIP_LINK_HERE"
ZIP_PATH = "/kaggle/working/dtmir_pro_plus.zip"

subprocess.run(["wget", "-q", "-O", ZIP_PATH, CODE_URL], check=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/kaggle/working')
print('extracted')

In [ ]:
import os
os.chdir('/kaggle/working/dtmir_pro_plus')
!python sanity_check.py

## 3. SOTS dehaze — download + prepare

In [ ]:
import pathlib, subprocess, sys

url = "https://huggingface.co/datasets/desonglll/reside-SOTS/resolve/main/SOTS.zip"
if not os.path.exists("SOTS.zip"):
    print("Downloading SOTS.zip (~435 MB)...")
    subprocess.run(["wget", "-q", "-O", "SOTS.zip", url], check=True)
if not os.path.exists("sots_raw"):
    subprocess.run(["unzip", "-q", "-o", "SOTS.zip", "-d", "sots_raw"], check=True)

base = pathlib.Path("sots_raw/SOTS")
split = "outdoor" if (base / "outdoor").is_dir() else "indoor"
print("Using SOTS split:", split)

r = subprocess.run([sys.executable, "-m", "data.prepare_sots",
                    "--sots-root", str(base / split), "--outdir", "data/sots",
                    "--size", "256", "--num", "400", "--seed", "0"],
                    capture_output=True, text=True)
print(r.stdout); print(r.stderr); r.check_returncode()

## 4. Train B1 (trainable translator) on SOTS dehaze — fast

In [ ]:
DATA_ROOT = 'data/sots'
EPOCHS_STAGE1 = 8
EPOCHS_STAGE2 = 20
IMAGE_SIZE = 256
BATCH_SIZE = 4

!python run_ablation.py --rows B1 --data-root {DATA_ROOT} --image-size {IMAGE_SIZE} \
    --batch-size {BATCH_SIZE} --epochs-stage1 {EPOCHS_STAGE1} \
    --epochs-stage2 {EPOCHS_STAGE2} --use-vgg 1 --outdir runs/ablation

## 5. SOTS dehaze — images + PSNR/SSIM/LPIPS/FID

In [ ]:
!python evaluate.py --checkpoint runs/ablation/B1/best_model.pt \
    --data-root data/sots/restoration --image-size 256 \
    --outdir runs/visual_sots --num 50 --grid-rows 6

## 6. Derain (Rain13K Test100) — download + prepare

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("dronefreak/Rain13K", repo_type="dataset",
                  allow_patterns=["test/Test100/*"], local_dir="rain_raw")

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "data.prepare_paired",
                    "--input-dir", "rain_raw/test/Test100/input",
                    "--target-dir", "rain_raw/test/Test100/target",
                    "--outdir", "data/rain", "--size", "256", "--num", "98",
                    "--identity-domain", "rain", "--seed", "0"],
                    capture_output=True, text=True)
print(r.stdout); print(r.stderr); r.check_returncode()

## 7. Train B1 on derain + visualize

In [ ]:
!python run_ablation.py --rows B1 --data-root data/rain --image-size 256 \
    --batch-size 4 --epochs-stage1 8 --epochs-stage2 20 --use-vgg 1 --outdir runs/rain

In [ ]:
!python evaluate.py --checkpoint runs/rain/B1/best_model.pt \
    --data-root data/rain/restoration --image-size 256 \
    --outdir runs/visual_rain --num 50 --grid-rows 6

## 8. Results to download

- `runs/visual_sots/qualitative.png` — dehaze comparison grid
- `runs/visual_sots/metrics.csv` — PSNR/SSIM/LPIPS/FID
- `runs/visual_rain/qualitative.png` — derain comparison grid
- `runs/visual_rain/metrics.csv`

Notes:
- SOTS is a ~400-pair subset, and Rain13K Test100 is 98 pairs, so numbers are a
  first visual result, not the full paper benchmark.
- The continuous-embedding change (B2/B4) is skipped here on purpose (it collapses
  without supervision); we're using B1 (the best config) for visuals.